# 🏠 Real Estate Investment Advisor
### ML-powered property profitability & future value predictor
**Domain:** Real Estate / Investment / Financial Analytics

This notebook covers:
1. Install dependencies
2. Generate & preprocess dataset
3. Exploratory Data Analysis (20 charts)
4. Model training — Classification & Regression
5. MLflow experiment tracking
6. Deploy Streamlit app via ngrok


## ⚙️ Step 0 — Install Libraries

In [ ]:
!pip install -q pandas numpy scikit-learn matplotlib seaborn mlflow xgboost streamlit pyngrok

## 📁 Step 1 — Upload Project Files

Upload all `.py` files from the project folder (generate_data.py, preprocess.py, eda.py, train_models.py, app.py).

In [ ]:
from google.colab import files
import os

os.makedirs('data', exist_ok=True)
os.makedirs('models', exist_ok=True)
os.makedirs('eda_plots', exist_ok=True)

uploaded = files.upload()   # Select all .py files
print('Uploaded:', list(uploaded.keys()))

## 🗃️ Step 2 — Generate & Preprocess Data

In [ ]:
exec(open('generate_data.py').read())
print('\nDataset created ✅')

In [ ]:
exec(open('preprocess.py').read())
df, label_encoders = load_and_preprocess()
df.to_csv('data/processed_data.csv', index=False)
print(f'Processed: {df.shape}  |  Good Investment ratio: {df["Good_Investment"].mean():.2%}')

## 📊 Step 3 — Exploratory Data Analysis

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
sns.set_theme(style='whitegrid')

# 1. Price distribution
fig, ax = plt.subplots(figsize=(10,4))
sns.histplot(df['Price_in_Lakhs'], bins=50, kde=True, color='steelblue', ax=ax)
ax.set_title('1. Distribution of Property Prices (Lakhs)', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# 2. Size distribution
fig, ax = plt.subplots(figsize=(10,4))
sns.histplot(df['Size_in_SqFt'], bins=50, kde=True, color='darkorange', ax=ax)
ax.set_title('2. Distribution of Property Sizes (SqFt)', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# 3. Price per sqft by property type
fig, ax = plt.subplots(figsize=(10,5))
order = df.groupby('Property_Type')['Price_per_SqFt'].median().sort_values(ascending=False).index
sns.boxplot(data=df, x='Property_Type', y='Price_per_SqFt', order=order, palette='Set2', ax=ax)
ax.set_title('3. Price per SqFt by Property Type', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# 4. Size vs Price
fig, ax = plt.subplots(figsize=(9,5))
ax.scatter(df['Size_in_SqFt'], df['Price_in_Lakhs'], alpha=0.3, color='teal', s=15)
ax.set_title('4. Property Size vs Price', fontsize=13, fontweight='bold')
ax.set_xlabel('Size (SqFt)'); ax.set_ylabel('Price (Lakhs)')
plt.tight_layout(); plt.show()

In [ ]:
# 5. Outlier detection
fig, axes = plt.subplots(1, 2, figsize=(12,5))
sns.boxplot(y=df['Price_per_SqFt'], color='salmon', ax=axes[0])
axes[0].set_title('Outliers: Price per SqFt')
sns.boxplot(y=df['Size_in_SqFt'], color='lightblue', ax=axes[1])
axes[1].set_title('Outliers: Property Size')
plt.suptitle('5. Outlier Detection', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# 6. Avg price per sqft by state
state_avg = df.groupby('State')['Price_per_SqFt'].mean().sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(11,5))
state_avg.plot(kind='bar', color='mediumseagreen', ax=ax)
ax.set_title('6. Avg Price per SqFt by State', fontsize=13, fontweight='bold')
plt.xticks(rotation=30, ha='right'); plt.tight_layout(); plt.show()

In [ ]:
# 7. Avg property price by city
city_avg = df.groupby('City')['Price_in_Lakhs'].mean().sort_values(ascending=False).head(15)
fig, ax = plt.subplots(figsize=(12,5))
city_avg.plot(kind='bar', color='cornflowerblue', ax=ax)
ax.set_title('7. Avg Property Price by City (Top 15)', fontsize=13, fontweight='bold')
plt.xticks(rotation=40, ha='right'); plt.tight_layout(); plt.show()

In [ ]:
# 8–20: Run remaining EDA from script
exec(open('eda.py').read())
run_eda(df)
print('All 20 EDA plots saved to eda_plots/')

## 🤖 Step 4 — Train Models & MLflow Tracking

In [ ]:
exec(open('train_models.py').read())
train_models()

## 🖥️ Step 5 — Launch Streamlit App (via ngrok)

In [ ]:
# Paste your ngrok auth token from https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_TOKEN = 'YOUR_NGROK_TOKEN_HERE'

from pyngrok import ngrok, conf
import subprocess, threading, time

conf.get_default().auth_token = NGROK_TOKEN

# Kill any existing ngrok tunnels
ngrok.kill()

def run_streamlit():
    subprocess.run(['streamlit', 'run', 'app.py',
                    '--server.port', '8501',
                    '--server.headless', 'true'])

t = threading.Thread(target=run_streamlit, daemon=True)
t.start()
time.sleep(4)

public_url = ngrok.connect(8501)
print('\n' + '='*55)
print(f'  🌐  Streamlit App: {public_url}')
print('='*55)
print('Open the link above in a new browser tab.')

## 📋 Step 6 — Manual Prediction (Without App)
Quick test of the trained model directly in the notebook.

In [ ]:
import pickle, numpy as np

with open('models/best_classifier.pkl','rb') as f: cls_bundle = pickle.load(f)
with open('models/best_regressor.pkl','rb')  as f: reg_bundle = pickle.load(f)
with open('models/scaler.pkl','rb')          as f: scaler = pickle.load(f)
with open('models/label_encoders.pkl','rb')  as f: le = pickle.load(f)
with open('models/feature_cols.pkl','rb')    as f: feature_cols = pickle.load(f)

# Example property
bhk=3; size=1500; price=95; transport=7; schools=5; hospitals=3
age=8; floor_no=5; total_floors=14; parking=2
infra = transport*0.4 + schools*0.3 + hospitals*0.3
school_den = schools*10 + hospitals*8
amenity_score = 4  # Gym+Pool
rera = 1
price_psf = (price*100000)/size

row = {
    'BHK':bhk,'Size_in_SqFt':size,'Price_in_Lakhs':price,'Price_per_SqFt':price_psf,
    'Age_of_Property':age,'Floor_No':floor_no,'Total_Floors':total_floors,
    'Floor_Ratio':floor_no/(total_floors+1),'Nearby_Schools':schools,
    'Nearby_Hospitals':hospitals,'Public_Transport_Accessibility':transport,
    'Parking_Space':parking,'Infra_Score':infra,'School_Density_Score':school_den,
    'Amenity_Score':amenity_score,'RERA_Ready':rera,
}

# Encode categorical defaults
defaults = {
    'State':'Karnataka','City':'Bangalore','Locality':'Koramangala',
    'Property_Type':'Apartment','Furnished_Status':'Semi-Furnished',
    'Facing':'North','Owner_Type':'Builder','Availability_Status':'Available',
    'Security':'Gated','Amenities':'Gym+Pool'
}
for col, val in defaults.items():
    enc = le[col]
    row[col+'_enc'] = int(enc.transform([val])[0]) if val in enc.classes_ else 0

X = np.array([row.get(c,0) for c in feature_cols]).reshape(1,-1)

cls_model = cls_bundle['model']
reg_model = reg_bundle['model']
X_c = scaler.transform(X) if cls_bundle.get('scaled') else X
X_r = scaler.transform(X) if reg_bundle.get('scaled') else X

pred_cls   = cls_model.predict(X_c)[0]
pred_proba = cls_model.predict_proba(X_c)[0][pred_cls]
pred_price = reg_model.predict(X_r)[0]
appreciation = ((pred_price - price) / price)*100

print('='*45)
print('  PROPERTY ANALYSIS RESULT')
print('='*45)
print(f'  Investment Decision : {"✅ GOOD INVESTMENT" if pred_cls else "❌ POOR INVESTMENT"}')
print(f'  Confidence          : {pred_proba*100:.1f}%')
print(f'  Current Price       : ₹{price:.1f} Lakhs')
print(f'  Estimated (5 Yrs)   : ₹{pred_price:.1f} Lakhs')
print(f'  Expected Gain       : +{appreciation:.1f}%')
print('='*45)